# Anti-DPO feasibility pilot and LR-LoRA probe

Run a 50-step standard-LoRA baseline before integrating or evaluating the experimental LR-LoRA adapter.

In [ ]:
from pathlib import Path
import importlib.util

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
required = ['torch', 'transformers', 'trl', 'peft', 'datasets', 'matplotlib']
missing = [name for name in required if importlib.util.find_spec(name) is None]
print('Missing:', missing)

In [ ]:
import subprocess, sys

command = [sys.executable, str(ROOT / 'scripts' / 'train.py'), '--dataset_path', str(ROOT / 'data' / 'russian_qa'), '--max_steps', '50']
print(' '.join(command))
RUN_TRAINING = False
if RUN_TRAINING:
    subprocess.run(command, check=True)

In [ ]:
if not missing:
    import sys, torch
    from torch import nn
    sys.path.insert(0, str(ROOT / 'src'))
    from lr_lora import LearnableRankLoRALinear
    probe = LearnableRankLoRALinear(nn.Linear(64, 64), rank=8, alpha=16, num_basis=8)
    with torch.no_grad():
        probe.nonlinearity.amplitudes.uniform_(-.1, .1)
    print('Stable rank:', float(probe.stable_rank()))
else:
    print('Install the training dependencies before executing the LR-LoRA probe.')